# Custom ChatGPT Application - Practice & Exercises

In this notebook, you will build a complete, stateful **Custom ChatGPT Clone** web application. This extends the Streamlit chatbot by adding a secure user registration and login system, local SQLite persistence, and chat log retrieval.

## Objectives
1. Implement a secure user authentication system (Registration, Login, and Logout) using SQLite and password hashing.
2. Link the active authenticated user's session to SQL-backed session storage (`SQLChatMessageHistory`).
3. Render user-specific history dynamically on login.
4. Provide a dashboard with chat capabilities, logout options, and sidebar controls (like clearing chat logs).

## Plan
1. **Database & Hashing Setup**: Establish an SQLite database (`chat_history.db`) with a `users` table and implement SHA-256 password hashing.
2. **User Authentication Flow**: Use `st.session_state` to track authentication status and the logged-in username, then build a landing portal where users can toggle between "Login" and "Register" forms.
3. **Dashboard & Session Initialization**: Once logged in, initialize `SQLChatMessageHistory` using the username as the session identifier.
4. **History Rendering & Chat Loop**: Retrieve and display past session messages, accept user chat input, save to SQLite, get the model's response, and save it back.
5. **Session Control & Logs**: Implement a "Log Out" button to clear session variables, and a "Clear Chat History" button to wipe chat logs.

### Step 1: Write the Complete Application Code

Run the following cell to write the complete Streamlit application code to `langchain/1_chat_models/7_chat_model_custom_chatgpt.py`.

In [ ]:
%%writefile ../../langchain/1_chat_models/7_chat_model_custom_chatgpt.py
import streamlit as st
import sqlite3
import hashlib
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from dotenv import load_dotenv

load_dotenv()

st.set_page_config(page_title="Custom ChatGPT App", page_icon="🤖")

DB_FILE = "chat_history.db"

# Database Helper Functions
def init_db():
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    # Create users table if not exists
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS users (
            username TEXT PRIMARY KEY,
            password TEXT NOT NULL
        )
    """)
    conn.commit()
    conn.close()

def hash_password(password):
    return hashlib.sha256(password.encode()).hexdigest()

def register_user(username, password):
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    try:
        cursor.execute("INSERT INTO users (username, password) VALUES (?, ?)", (username, hash_password(password)))
        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False
    finally:
        conn.close()

def login_user(username, password):
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute("SELECT password FROM users WHERE username = ?", (username,))
    row = cursor.fetchone()
    conn.close()
    if row and row[0] == hash_password(password):
        return True
    return False

# Initialize database
init_db()

# Session State Initialization
if "authenticated" not in st.session_state:
    st.session_state.authenticated = False
if "username" not in st.session_state:
    st.session_state.username = None

# Screen Navigation and Authentication Flow
if not st.session_state.authenticated:
    st.title("🔐 Custom ChatGPT Portal")
    
    # Toggle between Login and Register
    auth_mode = st.radio("Select Option:", ["Login", "Register"])
    
    username_input = st.text_input("Username:")
    password_input = st.text_input("Password:", type="password")
    
    if auth_mode == "Register":
        if st.button("Create Account"):
            if username_input and password_input:
                if register_user(username_input, password_input):
                    st.success("Registration successful! Please select 'Login' to continue.")
                else:
                    st.error("Username already exists. Please choose a different one.")
            else:
                st.warning("Please fill in all fields.")
                
    elif auth_mode == "Login":
        if st.button("Log In"):
            if username_input and password_input:
                if login_user(username_input, password_input):
                    st.session_state.authenticated = True
                    st.session_state.username = username_input
                    st.success("Login successful!")
                    st.rerun()
                else:
                    st.error("Invalid username or password.")
            else:
                st.warning("Please fill in all fields.")
else:
    # Authenticated Screen (Dashboard & Chat)
    st.title("🤖 Custom ChatGPT Dashboard")
    st.sidebar.header(f"👤 Account: {st.session_state.username}")
    
    # Logout action
    if st.sidebar.button("🔓 Log Out"):
        st.session_state.authenticated = False
        st.session_state.username = None
        st.rerun()
        
    # Setup SQL Message History unique to logged-in user
    chat_history = SQLChatMessageHistory(
        session_id=st.session_state.username,
        connection_string=f"sqlite:///{DB_FILE}"
    )

    # Sidebar Actions
    if st.sidebar.button("🗑️ Clear Chat History"):
        chat_history.clear()
        st.sidebar.warning("History cleared!")
        st.rerun()

    # Initialize model using core abstractions (Groq Llama model)
    model = init_chat_model("llama-3.3-70b-versatile", model_provider="groq")

    # If the SQL history is empty, write a default SystemMessage
    if len(chat_history.messages) == 0:
        chat_history.add_message(SystemMessage(content="You are a helpful and custom ChatGPT assistant."))

    # Render persisted chat history from SQLite
    for message in chat_history.messages:
        if isinstance(message, HumanMessage):
            with st.chat_message("user"):
                st.markdown(message.content)
        elif isinstance(message, AIMessage):
            with st.chat_message("assistant"):
                st.markdown(message.content)

    # User chat input
    if prompt := st.chat_input("Ask ChatGPT anything..."):
        # Render user message
        with st.chat_message("user"):
            st.markdown(prompt)
        # Save to SQLite database
        chat_history.add_user_message(prompt)

        # Get AI response
        with st.chat_message("assistant"):
            message_placeholder = st.empty()
            with st.spinner("Responding..."):
                response = model.invoke(chat_history.messages)
                message_placeholder.markdown(response.content)
        
        # Save response to SQLite database
        chat_history.add_ai_message(response.content)


### Step 2: How to Run the App

To run this Streamlit web application, execute the following command in your terminal (outside the Jupyter notebook environments):

```bash
streamlit run langchain/1_chat_models/7_chat_model_custom_chatgpt.py
```

### Exercise: Dynamic Model Swapper

#### Goal
Extend the sidebar options to include a model selector selectbox (`st.sidebar.selectbox`) that allows the logged-in user to swap between Llama (`llama-3.3-70b-versatile` via Groq) and Gemini (`gemini-2.5-flash` via Google GenAI) models dynamically without resetting the conversation history or logging out.

#### Solution
Implement the model swapper within the authenticated block of code. Click below to see the full code implementation of the solution.

<details>
<summary>💡 Click to see solution</summary>

```python
import streamlit as st
import sqlite3
import hashlib
from langchain_community.chat_message_histories import SQLChatMessageHistory
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from dotenv import load_dotenv

load_dotenv()

st.set_page_config(page_title="Custom ChatGPT App", page_icon="🤖")

DB_FILE = "chat_history.db"

# Database Helper Functions
def init_db():
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS users (
            username TEXT PRIMARY KEY,
            password TEXT NOT NULL
        )
    """)
    conn.commit()
    conn.close()

def hash_password(password):
    return hashlib.sha256(password.encode()).hexdigest()

def register_user(username, password):
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    try:
        cursor.execute("INSERT INTO users (username, password) VALUES (?, ?)", (username, hash_password(password)))
        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False
    finally:
        conn.close()

def login_user(username, password):
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    cursor.execute("SELECT password FROM users WHERE username = ?", (username,))
    row = cursor.fetchone()
    conn.close()
    if row and row[0] == hash_password(password):
        return True
    return False

# Initialize database
init_db()

# Session State Initialization
if "authenticated" not in st.session_state:
    st.session_state.authenticated = False
if "username" not in st.session_state:
    st.session_state.username = None

# Screen Navigation and Authentication Flow
if not st.session_state.authenticated:
    st.title("🔐 Custom ChatGPT Portal")
    
    # Toggle between Login and Register
    auth_mode = st.radio("Select Option:", ["Login", "Register"])
    
    username_input = st.text_input("Username:")
    password_input = st.text_input("Password:", type="password")
    
    if auth_mode == "Register":
        if st.button("Create Account"):
            if username_input and password_input:
                if register_user(username_input, password_input):
                    st.success("Registration successful! Please select 'Login' to continue.")
                else:
                    st.error("Username already exists. Please choose a different one.")
            else:
                st.warning("Please fill in all fields.")
                
    elif auth_mode == "Login":
        if st.button("Log In"):
            if username_input and password_input:
                if login_user(username_input, password_input):
                    st.session_state.authenticated = True
                    st.session_state.username = username_input
                    st.success("Login successful!")
                    st.rerun()
                else:
                    st.error("Invalid username or password.")
            else:
                st.warning("Please fill in all fields.")
else:
    # Authenticated Screen (Dashboard & Chat)
    st.title("🤖 Custom ChatGPT Dashboard")
    st.sidebar.header(f"👤 Account: {st.session_state.username}")
    
    # Logout action
    if st.sidebar.button("🔓 Log Out"):
        st.session_state.authenticated = False
        st.session_state.username = None
        st.rerun()
        
    # Model Selection
    selected_model = st.sidebar.selectbox(
        "Select Model Provider:",
        ["Llama 3.3 (Groq)", "Gemini 2.5 (Google)"]
    )
    
    if selected_model == "Llama 3.3 (Groq)":
        model_name, provider = "llama-3.3-70b-versatile", "groq"
    else:
        model_name, provider = "gemini-2.5-flash", "google_genai"

    # Setup SQL Message History unique to logged-in user
    chat_history = SQLChatMessageHistory(
        session_id=st.session_state.username,
        connection_string=f"sqlite:///{DB_FILE}"
    )

    # Sidebar Actions
    if st.sidebar.button("🗑️ Clear Chat History"):
        chat_history.clear()
        st.sidebar.warning("History cleared!")
        st.rerun()

    # Dynamic Model loading using core abstractions
    model = init_chat_model(model_name, model_provider=provider)

    # If the SQL history is empty, write a default SystemMessage
    if len(chat_history.messages) == 0:
        chat_history.add_message(SystemMessage(content="You are a helpful and custom ChatGPT assistant."))

    # Render persisted chat history from SQLite
    for message in chat_history.messages:
        if isinstance(message, HumanMessage):
            with st.chat_message("user"):
                st.markdown(message.content)
        elif isinstance(message, AIMessage):
            with st.chat_message("assistant"):
                st.markdown(message.content)

    # User chat input
    if prompt := st.chat_input("Ask ChatGPT anything..."):
        # Render user message
        with st.chat_message("user"):
            st.markdown(prompt)
        # Save to SQLite database
        chat_history.add_user_message(prompt)

        # Get AI response
        with st.chat_message("assistant"):
            message_placeholder = st.empty()
            with st.spinner("Responding..."):
                response = model.invoke(chat_history.messages)
                message_placeholder.markdown(response.content)
        
        # Save response to SQLite database
        chat_history.add_ai_message(response.content)
```
</details>